# Text Classification

## Importing required libraries

In [1]:
import pandas as pd

## Loading the Data
source: https://www.kaggle.com/ashishpatel26/sentimental-analysis-nlp

In [2]:
df = pd.read_csv('Datasets/sentiment_data.csv', header=None, names=['Label', 'Text'], sep='\t')

## Data Cleaning and Preparation

### Check the dimensions 

In [3]:
df.shape

(6918, 2)

### Check columns names

In [4]:
df.columns

Index(['Label', 'Text'], dtype='object')

In [5]:
df.sample(10)

,Label,Text
2316,1,Harry Potter is AWESOME I don't care if anyone...
2228,1,I love Harry Potter.
4536,0,friday hung out with kelsie and we went and sa...
3874,1,I love Brokeback Mountain....
5457,0,This quiz sucks and Harry Potter sucks ok bye..
2681,1,"I, too, like Harry Potter.."
3939,1,dudeee i LOVED brokeback mountain!!!!
5577,0,This quiz sucks and Harry Potter sucks ok bye..
3419,1,dudeee i LOVED brokeback mountain!!!!
2052,1,Write us a Harry Potter poem for a chance to w...


### Check for Nulls

In [6]:
df.isnull().values.any()

False

## Machine Learning: LinearSVC

In [7]:
import sklearn
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score

### Define Output and Inputs

In [8]:
y = df['Label']
X = df['Text']

### Split dataset

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [10]:
X_train.shape, X_test.shape

((5534,), (1384,))

In [11]:
y_train.shape, y_test.shape

((5534,), (1384,))

### Vectorizer

In [12]:
tfidf_vect = TfidfVectorizer(max_features=15)

X_trans= tfidf_vect.fit_transform(X_train)

### Instantiate LinearSVC

In [13]:
classifier = LinearSVC(C=1.0, max_iter=1000, tol=1e-3)

linear_svc_model = classifier.fit(X_trans, y_train)

In [14]:
X_test_trans = tfidf_vect.fit_transform(X_test)

In [15]:
y_pred = linear_svc_model.predict(X_test_trans)

### Score

In [16]:
pred_results = pd.DataFrame({'y_test': y_test, 'y_pred': y_pred})

In [17]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.9002890173410405

### Pipeline

In [18]:
from sklearn.pipeline import Pipeline

In [19]:
clf_pipeline = Pipeline(steps=[('tfidf_vect', tfidf_vect), ('classifier', classifier)])

pipeline_model = clf_pipeline.fit(X_train, y_train)

In [20]:
y_pred = pipeline_model.predict(X_test) #can use data directly

In [21]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.9002890173410405

### Create a Checkpoint

In [22]:
pipe_clf_param = {}

pipe_clf_param['pipeline_clf'] = pipeline_model
pipe_clf_param['sklearn_version'] = sklearn.__version__
pipe_clf_param['accuracy'] = accuracy

pipe_clf_param

{'pipeline_clf': Pipeline(steps=[('tfidf_vect', TfidfVectorizer(max_features=15)),
                 ('classifier', LinearSVC(tol=0.001))]),
 'sklearn_version': '0.24.2',
 'accuracy': 0.9002890173410405}

### Save

In [23]:
import joblib

In [24]:
filename = 'Models/pipe_clf_model_checkpoint.joblib'

In [25]:
joblib.dump(pipe_clf_param, filename)

['Models/pipe_clf_model_checkpoint.joblib']

### Load

In [26]:
pipe_clf_checkpoint = joblib.load(filename)

In [27]:
reloaded_pipeline = pipe_clf_checkpoint['pipeline_clf']

### Test

In [28]:
y_pred = reloaded_pipeline.predict(X_test)

In [29]:
accuracy_score(y_test, y_pred)

0.9002890173410405